# Field dependence over the whole zone: Figures 7 and 8

$\eta^{x;xx}(\hbar\omega, B)$ over the whole Brillouin zone for the TNN model (GGA) with all three field terms, and which term does what (Sec. III B).

## Load engine

In [ ]:
%run engine.ipynb

## Settings

In [ ]:
N = 600            # k-mesh over the zone (150 for a quick look)
n_omega = 900      # photon energies in the maps (300 for a quick look)
dB_map = 1         # field step in the maps, in T (10 for a quick look)
R_disk = 0.9       # disk radius around each valley (1/a)
h_dec = 0.002      # disk spacing for the decomposition (0.005 for a quick look; 0.01 gives wrong signs)

zone = slanted_grid(N)

## The field terms at the valleys

MoS$_2$ (TNN) at $B = 30$ T, at the band edge of each valley (spin up at $K$, spin down at $K'$): the moments of the two bands, how far the energy shift moves the gap, and the phase-space factor of the valence band.

In [ ]:
B = 30.0
p = params("MoS2")
for valley, k_x0, block in [("K", K_len, 0), ("K'", -K_len, 1)]:
    k_x, k_y = np.array([k_x0]), np.array([0.0])
    H = lambda x, y: zeeman(hamiltonian("TNN", p, x, y), lambda_SOC["MoS2"], B)[block]
    E, V, v_x, v_y = bands(H, k_x, k_y)
    Omega, M = berry_and_moment(E, v_x, v_y, p["a"])
    E_res, dE_dkx, dE_dky, D_v = field_terms(H, k_x, k_y, E, v_x, v_y, Omega, M, p["a"], B, B, B)
    gap_shift = (E_res[0, 1] - E_res[0, 0]) - (E[0, 1] - E[0, 0])
    print(f"At {valley}: M_v = {M[0, 0]:.3f} mu_B, M_c = {M[0, 1]:.3f} mu_B, "
          f"the energy shift moves the gap by {1e3 * gap_shift:+.3f} meV, D_v = {D_v[0]:.4f}")

## Maps $\eta^{x;xx}(\hbar\omega, B)$

$\nu = 10$ meV, $\hbar\omega$ from $0.95E_g$ to $1.05E_{\max}$. We only compute $B \ge 0$: time reversal gives $\eta(-B) = -\eta(B)$. At paper settings this is about 30 s per field, 2 h for the four materials.

In [ ]:
B_map = np.arange(0, 60 + 1e-9, dB_map)
for m in materials:
    file = f"data/map_{m}_N{N}_n{n_omega}_dB{dB_map}.npz"
    if os.path.exists(file):
        continue
    omega = np.linspace(0.95 * gap(m), 1.05 * E_max(m), n_omega)
    eta = []
    for B in B_map:
        eta.append(eta_mag(m, "TNN", B, omega, 0.01, zone)[:, 0, 0, 0])
    np.savez(file, B=B_map, omega=omega, eta=eta_SI(params(m)["a"]) * np.array(eta))

maps = {m: np.load(f"data/map_{m}_N{N}_n{n_omega}_dB{dB_map}.npz") for m in materials}
for m in materials:
    print(f"{m}: largest |eta(B = 0)| = {np.abs(maps[m]['eta'][0]).max():.1e}, largest |eta| = {np.abs(maps[m]['eta']).max():.3f} A m V^-2 s^-1")

## Figure 7: $\eta^{x;xx}(B)$ at the van Hove energy

The van Hove energy is where $|\eta|$ is largest at 60 T. The peak is flat, so $\chi$ there moves by about 10% between neighbouring photon energies. The small-field slope is $\chi = \eta(B_1)/B_1$ at the smallest field in the map, $B_1 = 1$ T. Negative fields come from `mirror`.

In [ ]:
fig, ax = plt.subplots(figsize=(CW, 2.45))
vH = {}
for m in materials:
    d = maps[m]
    j = np.argmax(np.abs(d["eta"][-1]))
    vH[m] = d["omega"][j]
    B_all, eta_all = mirror(d["B"], d["eta"][:, j])
    ax.plot(B_all, eta_all, color=mat_color[m], label=mat_label[m] + r" ($%.2f$ eV)" % vH[m])
    slope = d["eta"][1, j] / d["B"][1]
    below = 100 * (1 - eta_all[-1] / (slope * 60))
    print(f"{m}: van Hove energy {vH[m]:.3f} eV, small-field slope = {slope:.4f} A m V^-2 s^-1 T^-1, eta(60 T) = {eta_all[-1]:.3f}, {below:.1f}% below linear at 60 T")
zero(ax, v=True)
ax.set_xlabel(r"$B$ (T)")
ax.set_ylabel(r"$\eta^{x;xx}$ ($%s$)" % UNIT)
ax.set_xlim(-62, 62)
ax.legend(loc="upper left")
ax.xaxis.set_major_locator(MaxNLocator(7, steps=[1, 2, 2.5, 5, 10]))
plt.savefig("figures/lpge_field_merged.pdf")
plt.show()

## Figure 8: maps

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(TW, 1.95), gridspec_kw=dict(wspace=0.62))
for j, m in enumerate(materials):
    d = maps[m]
    ax = axes[j]
    B_all, eta_all = mirror(d["B"], d["eta"])
    biggest = np.abs(eta_all).max()
    im = ax.pcolormesh(d["omega"], B_all, eta_all, cmap="RdBu_r", vmin=-biggest, vmax=biggest, shading="nearest", rasterized=True)
    cb = fig.colorbar(im, ax=ax, pad=0.03, fraction=0.07, aspect=22)
    cb.ax.tick_params(labelsize=6, width=0.5, length=2, direction="out")
    cb.outline.set_linewidth(0.5)
    cb.ax.minorticks_off()
    cb.locator = MaxNLocator(6, symmetric=True)
    cb.update_ticks()
    if j == 3:
        cb.set_label(r"$\eta^{x;xx}$ ($%s$)" % UNIT, fontsize=7, labelpad=2)
    ax.set_xlabel(r"$\hbar\omega$ (eV)")
    ax.set_title(mat_label[m], pad=3, loc="right")
    label(ax, f"({'abcd'[j]})")
    ax.tick_params(top=False, right=False, which="both")
    ax.xaxis.set_major_locator(MaxNLocator(4))
    ax.set_ylim(B_all.min(), B_all.max())
    ax.yaxis.set_major_locator(MaxNLocator(5, steps=[1, 2, 3, 5, 6, 10]))
axes[0].set_ylabel(r"$B$ (T)")
plt.savefig("figures/lpge_map_grid.pdf")
plt.show()

## Which field term does what (Sec. III B)

$\chi = \eta^{x;xx}(10\,{\rm T})/10$ T with the field terms switched on one at a time, on top of the Zeeman Hamiltonian: Z = Zeeman only, D = phase-space factor, E = energy shift, V = velocity shift, F = all three. At the van Hove energy on the whole zone ($\nu = 10$ meV) and at $E_g + 0.1$ eV on the valley disks ($\nu = 5$ meV). About 15 min per material.

In [ ]:
variants = {"Z": (False, False, False), "D": (False, False, True), "E": (True, False, False),
            "V": (False, True, False), "F": (True, True, True)}          # (energy shift, velocity shift, phase-space factor)
valley_disk = disk(R_disk, h_dec)

for m in materials:
    file = f"data/decomp_{m}_N{N}_n{n_omega}_dB{dB_map}_R{R_disk}_h{h_dec}.npz"
    if os.path.exists(file):
        continue
    si = eta_SI(params(m)["a"])
    out = {}
    for name, (E_on, V_on, D_on) in variants.items():
        eta_vH = eta_mag(m, "TNN", 10.0, np.array([vH[m]]), 0.01, zone,
                         energy_shift=E_on, velocity_shift=V_on, phase_space=D_on)
        eta_edge = eta_mag(m, "TNN", 10.0, np.array([gap(m) + 0.1]), 0.005, valley_disk, centers=[K_len, -K_len],
                           energy_shift=E_on, velocity_shift=V_on, phase_space=D_on)
        out["vH_" + name] = si * eta_vH[0, 0, 0, 0] / 10
        out["edge_" + name] = si * eta_edge[0, 0, 0, 0] / 10
    np.savez(file, **out)

print("chi in 1e-2 (van Hove) and 1e-4 (band edge) A m V^-2 s^-1 T^-1")
print("           " + "".join(f"{name:>9s}" for name in variants) + "     F/Z")
for m in materials:
    d = np.load(f"data/decomp_{m}_N{N}_n{n_omega}_dB{dB_map}_R{R_disk}_h{h_dec}.npz")
    print(f"{m:6s} vH  " + "".join(f"{1e2 * d['vH_' + name]:+9.3f}" for name in variants) + f"  {d['vH_F'] / d['vH_Z']:6.2f}")
    print(f"{m:6s} edge" + "".join(f"{1e4 * d['edge_' + name]:+9.3f}" for name in variants) + f"  {d['edge_F'] / d['edge_Z']:6.2f}")

## Convergence in $N$

$\chi = \eta^{x;xx}(1\,{\rm T})/1$ T at the van Hove energy, on the paper's $N = 600$ mesh and on $N = 900$. About 1 min per run.

In [ ]:
file = "data/convergence_N.npz"
if not os.path.exists(file):
    out = {}
    for m in materials:
        for N_conv in (600, 900):
            out[f"{m}_{N_conv}"] = eta_SI(params(m)["a"]) * eta_mag(m, "TNN", 1.0, np.array([vH[m]]), 0.01, slanted_grid(N_conv))[0, 0, 0, 0]
    np.savez(file, **out)
d = np.load(file)
for m in materials:
    print(f"{m}: chi(N = 600) = {d[m + '_600']:.5f}, chi(N = 900) = {d[m + '_900']:.5f}, change {100 * (d[m + '_600'] / d[m + '_900'] - 1):+.3f}%")